<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

These are the libraries used below. Run the cell once in the environment where OpenBB is already installed, then restart the kernel so the imports resolve.

In [ ]:
!pip install numpy pandas matplotlib scikit-learn-extra

## Imports and setup

numpy handles the annualization math and pandas reads the Wikipedia table and reshapes the price data. matplotlib draws the scatter plot, KMedoids from scikit-learn-extra groups the stocks, and openbb pulls daily closes for every ticker in one call.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn_extra.cluster import KMedoids
import yfinance as yf

## Pull Nasdaq-100 members and prices

yFinance takes the whole list in one request and aligns the series on shared dates.

In [ ]:
nasdaq_stocks = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "TSLA", "AVGO", "COST", "NFLX"]

data = yf.download(
    nasdaq_stocks,
    start="2020-01-01",
    end="2022-12-31",
).Close

data comes back as a DataFrame with one column per ticker and one row per trading day, so data.shape tells you how many series actually returned and whether any symbol came back empty. A name added to the index in 2022 carries NaNs through the earlier rows, and pct_change passes those straight into the next step.

## Measure yearly return and daily swings

pct_change turns prices into daily percentage changes, and describe gives the mean and the standard deviation of those changes per stock, which the multiplier then scales to a yearly figure.

In [ ]:
moments = (
    data
    .pct_change()
    .describe()
    .T[["mean", "std"]]
    .rename(columns={"mean": "returns", "std": "vol"})
) * [252, np.sqrt(252)]

Multiplying the mean by 252 and the standard deviation by the square root of 252 gives annual figures, since there are about 252 trading days in a year. That square root step assumes daily returns are roughly independent from one day to the next, which holds less well during a crash month. The matrix passed to the model has only these two columns, so sector never enters the fit. Run moments.describe() and read the min and max of each column, because the distance calculation works on these raw numbers and nothing standardizes them first.

## Group the stocks and plot results

KMedoids with n_clusters=5 assigns every stock to one of five groups, labels_ holds those assignments, and Spectral supplies one color per group.

In [ ]:
km = KMedoids(n_clusters=5).fit(moments)
labels = km.labels_
unique_labels = set(labels)
colors = [
    plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))
]

Each group center is an actual stock rather than an average point, so under the default Euclidean distance one extreme name moves the center less than it would in KMeans, though the size of that difference only shows up if you fit both on moments and compare. Refit with 4 and 6 clusters and score the label sets against each other with adjusted_rand_score, which returns 1 when two runs put the same stocks together. Five is illustrative, not a count any validation score picked.

The scatter plot carries no ticker labels, so print moments.index[km.medoid_indices_] to read the five names drawn in cyan and match each one to its group.

In [ ]:
for k, col in zip(unique_labels, colors):
    class_member_mask = labels == k
    xy = moments[class_member_mask]
    plt.plot(
        xy.iloc[:, 0],
        xy.iloc[:, 1],
        "o",
        markerfacecolor=tuple(col),
        markeredgecolor="k",
    )

In [ ]:
plt.plot(
    km.cluster_centers_[:, 0],
    km.cluster_centers_[:, 1],
    "o",
    markerfacecolor="cyan",
    markeredgecolor="k",
)
plt.xlabel("Return")
plt.ylabel("Ann. Vol.")
plt.show()

Outliers are easier to pick with a rule than by eye. Sort moments by vol and by returns, take any row more than three standard deviations from its column mean, drop it, and refit. Compare the new labels against the old with adjusted_rand_score, and a value below roughly 0.7 means one ticker was driving the split. Within a group, the next step is the correlation matrix of daily returns, which is what tells you whether those names actually move together.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.